# Minero GPU en Google Colab

Suma la GPU de Colab (una T4 en el plan gratis) al pool del sistema desplegado en GKE, junto a la del k3s. Es el mismo minero de `pilar2/minero_gpu`: sólo hace pedidos HTTP salientes al gateway.

1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.
2. Correr las celdas en orden. La última queda minando; para sacar la GPU del pool, detenerla (el keep-alive vence a los 15 s).

In [ ]:
# 1. Hay GPU y de qué arquitectura es
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

In [ ]:
# 2. El código y el binario CUDA (compilado para la GPU que tocó)
!rm -rf /content/BlockChain-Distribuida
!git clone -q --depth 1 https://github.com/chimera-sdypp/BlockChain-Distribuida /content/BlockChain-Distribuida
%cd /content/BlockChain-Distribuida
!ARQ=sm_$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .) && \
  echo "compilando para $ARQ" && \
  nvcc -O3 -arch=$ARQ -o /content/minar pilar2/minero_gpu/minar.cu
!pip install -q -r pilar2/comun/requirements.txt

In [ ]:
# 3. Configuración: la URL del gateway y el id se completan acá; el token se pide aparte para no guardarlo en el notebook
import os, getpass
GATEWAY_URL = ""  #@param {type:"string"}
MINERO_ID = "gpu-colab-"  #@param {type:"string"}
assert GATEWAY_URL.startswith("http"), "falta GATEWAY_URL (http://<IP>:8081)"
assert MINERO_ID not in ("", "gpu-colab-"), "poné un MINERO_ID único, p. ej. gpu-colab-mateo"
os.environ.update(GATEWAY_URL=GATEWAY_URL, MINERO_ID=MINERO_ID, MINAR="/content/minar")
os.environ["MINER_TOKEN"] = getpass.getpass("MINER_TOKEN: ")
!curl -s -m 10 $GATEWAY_URL/health; echo

In [ ]:
# 4. Minar. Detener la celda saca la GPU del pool.
%cd /content/BlockChain-Distribuida/pilar2
!python -m minero_gpu